In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
data=pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
data.info()

In [ ]:
data.sample(10)

In [ ]:
data.drop(['customerID'], axis=1, inplace=True)

In [ ]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

In [ ]:
data.dropna(subset=['TotalCharges'], inplace=True)
data.reset_index(drop=True, inplace=True)

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
plt.figure(figsize=(6,5))

ax = sns.countplot(data=data, x='Churn')

for container in ax.containers:
    ax.bar_label(container)

plt.title("Churn Distribution")
plt.show()

print(data['Churn'].value_counts(normalize=True)*100)

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(len(num_cols), 2, figsize=(12,12))

for i, col in enumerate(num_cols):

    sns.histplot(data[col], kde=True, ax=axes[i,0])
    axes[i,0].set_title(f'Histogram of {col}')

    sns.boxplot(x=data[col], ax=axes[i,1])
    axes[i,1].set_title(f'Boxplot of {col}')

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = data.select_dtypes(include='object').columns.drop('Churn')

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(9, 4*len(cat_cols)))

if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    sns.countplot(data=data, x=col, order=data[col].value_counts().index, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 12))

for i, col in enumerate(num_cols):

    sns.kdeplot(
        data=data,
        x=col,
        hue='Churn',
        fill=True,
        common_norm=False,
        ax=axes[i,0]
    )
    axes[i,0].set_title(f'{col} Distribution by Churn')

    sns.boxplot(
        data=data,
        x='Churn',
        y=col,
        ax=axes[i,1]
    )
    axes[i,1].set_title(f'{col} vs Churn')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

corr = data[['SeniorCitizen',
             'tenure',
             'MonthlyCharges',
             'TotalCharges']].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
for col in data.select_dtypes(include='object').columns:
    print("=" * 40)
    print(col)
    print(data[col].value_counts())

In [ ]:
data[['tenure','MonthlyCharges','TotalCharges']].skew()

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = data[(data[col] < lower) | (data[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

In [ ]:
binary_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
    'Churn'
]

mapping = {
    'Yes':1,
    'No':0,
    'Male':1,
    'Female':0
}

for col in binary_cols:
    data[col] = data[col].map(mapping)

In [ ]:
multi_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

data = pd.get_dummies(
    data,
    columns=multi_cols,
    drop_first=True,
    dtype=int
)

In [ ]:
data.info()

In [ ]:
for col in data.select_dtypes(include='object').columns:
    print(f"{col}: {data[col].unique()}")
    print("-"*50)

In [ ]:
X = data.drop('Churn', axis=1)
y = data['Churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(random_state=42)

lr.fit(X_train, y_train)
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

y_pred = lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))


cv = cross_val_score(
    lr,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:", cv)
print("Mean Accuracy:", cv.mean())

In [ ]:
params = {
    'C':[0.01,0.1,1,10,100],
    'solver':['liblinear','lbfgs']
}

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train,y_train)

print(grid_lr.best_params_)
print(grid_lr.best_score_)

In [ ]:
knn = KNeighborsClassifier()

knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print()
print(classification_report(y_test, y_pred_knn))


cv_knn = cross_val_score(
    knn,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:")
print(cv_knn)
print("Mean Accuracy:", cv_knn.mean())

In [ ]:
param_knn = {
    'n_neighbors':[3,5,7,9,11,13],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Best Parameters:")
print(grid_knn.best_params_)

print()

print("Best CV Score:")
print(grid_knn.best_score_)

In [ ]:
best_knn = grid_knn.best_estimator_

y_pred_best_knn = best_knn.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best_knn))
print()

print(classification_report(y_test, y_pred_best_knn))

In [ ]:
svm = SVC(random_state=42)

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print()

print(classification_report(y_test, y_pred_svm))


cv_svm = cross_val_score(
    svm,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:")
print(cv_svm)
print("Mean Accuracy:", cv_svm.mean())

In [ ]:
param_svm = {
    'C':[0.1,1,10],
    'kernel':['linear','rbf'],
    'gamma':['scale','auto']
}

grid_svm = GridSearchCV(
    SVC(random_state=42),
    param_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Best Parameters:")
print(grid_svm.best_params_)
print("Best CV Score:")
print(grid_svm.best_score_)

In [ ]:
best_svm = grid_svm.best_estimator_

y_pred_best_svm = best_svm.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best_svm))
print()

print(classification_report(y_test, y_pred_best_svm))

In [ ]:

results = {
    "Logistic Regression": accuracy_score(y_test, lr.predict(X_test)),
    "KNN": accuracy_score(y_test, best_knn.predict(X_test)),
    "SVM": accuracy_score(y_test, best_svm.predict(X_test))
}

print(results)

In [ ]:
plt.figure(figsize=(7,5))

plt.bar(results.keys(), results.values())

plt.ylim(0.7,1)

plt.ylabel("Accuracy")

plt.title("Model Comparison")

plt.show()